## Homework 3 - Analysis Using the $p$, $np$, $c$, and $u$ Control Charts
### Dr. Austin Brown
### Kennesaw State University

**DUE: October 30, 2026 via D2L**

## Instructions:

Ravi's Comics and Games is a local comic book and tabletop game shop that receives weekly new-release shipments from a regional distributor. Ravi's customers, especially the collectors chasing near-mint copies, have started complaining more often about comics arriving with scratched or folded covers, and Ravi has noticed it's starting to hurt sales of his higher-end back issues too. He's asked your quality control team to help him figure out how bad the problem really is, whether it's getting better or worse, and who -- the distributor, or his own shop -- is responsible.

Your team decides to look at this two different ways. First, every comic in a week's shipment gets a quick pass/fail condition check as staff unbox and shelve it (any visible scratch, crease, or fold on the cover marks it nonconforming). Because the number of comics in a shipment varies week to week depending on the new-release schedule, your team tracks both the **rate** of nonconforming comics (useful for spotting trends) and the **raw count** (useful when Ravi calls the distributor's account representative, who thinks in terms of "how many books," not percentages).

Second, your team recognizes that a single comic can have more than one distinct defect on it -- a scratch on the front cover **and** a folded corner on the back, for example -- so a simple pass/fail check doesn't capture the full severity of the damage. To get a fuller picture, staff also perform a more detailed defect count each week, counting every individual scratch, fold, or dent they find (not just how many comics have at least one). This detailed count is done on two separate parts of the shop: a **fixed** sample of 5 boxes pulled from the new-release shipment every week, and a **variable** number of already-shelved back-issue boxes, inspected whenever staff have the time.

## Key Specifications

| Quantity | Value |
|---|---|
| Nonconforming definition (Datasets A) | Any visible scratch, crease, or fold on the cover |
| Nonconformity definition (Datasets B & C) | Each individual scratch, fold, or dent found (a comic may have more than one) |
| Dataset A: Shipment Condition, Phase I weeks (m) | 25 weeks, variable shipment size |
| Dataset A: Shipment Condition, Phase II weeks (m) | 20 weeks, variable shipment size |
| Dataset B: New-Release Defect Counts, weeks (m) | 25 weeks, fixed sample of 5 boxes/week |
| Dataset C: Back-Issue Defect Counts, weeks (m) | 25 weeks, variable number of boxes/week |

## Define & Measure

### Q1: Briefly describe the purpose of this analysis
The purpose of this analysis is monitoring the condition of comics and determining whether the quality of shipments is improving or worsening. We also need to identify whether defects are coming from the distributor or Ravi's shop.

### Q2: Ravi's team is monitoring two distinct quality characteristics here: a pass/fail condition check, and a detailed defect count. For each, classify the type of attribute data being collected, and briefly explain how this differs from the continuous, normally-distributed data used in your previous two homeworks.

The pass-fail condition check is binomial because each comic is either conforming or nonconforming. It's different from the past two homeworks where we had continuous data in the form of measurements and was modeled as a normal distribution. Here we can use poisson distribution since the comic can have multiple defects.

### Q3: For the pass/fail data (Dataset A), what practical factor determines whether you should build a $p$ chart or an $np$ chart? For the detailed defect-count data (Datasets B and C), what practical factor determines whether you should build a $c$ chart or a $u$ chart?

For pass/fail data we choose between a p chart and an np chart depending on whether the sample size varies or stays constant. So we use p and np charts for dataset A where p chart monitors the proportion nonconforming and works with variable sample sizes, and a np chart monitors the number nonconforming and is used with constant sample sizes. For dataset B we use a c chart to monitor defect counts when the inspection size is constant. For dataset C we will use a u chart for whe the inspection size varies.

## Analyze

### Q4: Read the Phase I Shipment Condition data (Dataset A) into R. Construct a Phase I $p$ chart. Does the chart show evidence of statistical control? If not, identify the out-of-control week(s) by number and provide a contextual explanation as to why these OOC point(s) may have been observed, omit them, and replot the chart until statistical control is observed.

The original p chart shows that the process is not statistically in control because week 14 falls above the upper control limit. We suspect this could have been caused by a damaged shipment from the distributor, or rough handling from distribution or delivery staff. After removing week 14 and recalculating the control limits, all remaining weeks fall within the limits which indicates that the process is in statistical control.

In [ ]:
## Q4 Code ##

library(tidyverse)
library(qcc)
library(readxl)

# data
phase1 <- read_excel("C:\\Users\\diirt\\Desktop\\STAT7110\\HW3\\RavisComics_ShipmentCondition_PhaseI.xlsx")

# original p chart
p1 <- qcc(phase1$NumberNonconforming,
          sizes = phase1$TotalComics,
          type = "p")

# remove Week 14
phase1_new <- phase1 %>%
  filter(Week != 14)

# update p chart
p2 <- qcc(phase1_new$NumberNonconforming,
          sizes = phase1_new$TotalComics,
          type = "p")


### Q5: Repeat the analysis in Q4, this time constructing an $np$ chart on the same data. Do your conclusions about which week(s) are out of control change? Which chart do you think would be easier to explain to the distributor's account representative, and why?

The np chart identifies week 14 as out of control which is consistent with our p chart. After removing week 14, the rest of the weeks are within the control limits. I think the np chart is easier to explain to the distributor's rep because it shows the actual quantity of damaged comics rather than percentages. The p chart might be better for monitoring the process because shipment sizes vary week by week.

In [ ]:
## Q5 Code ##

# original np chart
np1 <- qcc(phase1$NumberNonconforming,
           sizes = phase1$TotalComics,
           type = "np")

# update np chart w/o week 14
np2 <- qcc(phase1_new$NumberNonconforming,
           sizes = phase1_new$TotalComics,
           type = "np")


### Q6: Using your finalized Phase I estimate of $\bar{p}$, report the center line and control limits you will carry forward into Phase II.

The final phase1 center line is 0.05162 which means approximately 5.16% of comics are expected to be nonconforming. The upper and lower control limits depend on the number of comics inspected each week, so these finalized phase1 estimates will be used to evaluate whether the process remains statistically in control during phase2. the controll imits will be calculated using each week's sample size.

In [ ]:
## Q6 Code ##

# final phase1 center line
pbar <- sum(phase1_new$NumberNonconforming) /
        sum(phase1_new$TotalComics)

# control limits
LCL <- pmax(0, pbar - 3 * sqrt(
  pbar * (1 - pbar) / phase1_new$TotalComics))

UCL <- pbar + 3 * sqrt(
  pbar * (1 - pbar) / phase1_new$TotalComics)

# results
pbar
LCL
UCL


## Improve

### Q7: After seeing your Phase I results, Ravi calls the distributor, who agrees to add cardboard stiffeners to every shipping box going forward. Briefly explain why it would not be appropriate to use your *original* (uncleaned) Phase I control limits, rather than the finalized limits from Q6, to judge whether this packaging fix actually worked.

The original phase1 control limits included week14 which we identified as out of control. Including that observation would distort the estimated process average and control limits, which makes it harder to determine whether the packaging improvement is effective. The final limits provide a more accurate baseline of the process before the cardboard stiffeners were introduced.

## Control

### Q8: Read the Phase II Shipment Condition data into R. Using your finalized Phase I center line and control limits from Q6, construct the Phase II $p$ chart. Do any weeks plot outside the control limits? Does this support the conclusion that the packaging fix worked?

The phase2 p chart shows that week2 falls below the lower control limit, while the rest of the weeks are within the limits. This suggests that the process have improved following the introduction of cardboard stiffeners. The overall proportion of nonconforming comics has decreased from 5.16% in phase1 to 2.71% in phase2. The point below the lower control limit provides evidence of a positive change in the process.

In [ ]:
## Q8 Code ##

# data
phase2 <- read_excel("C:\\Users\\diirt\\Desktop\\STAT7110\\HW3\\RavisComics_ShipmentCondition_PhaseII.xlsx")

# final phase1 center line
pbar <- 0.05162

# phase2 p chart
p_phase2 <- qcc(phase2$NumberNonconforming,
                sizes = phase2$TotalComics,
                type = "p",
                center = pbar,
                std.dev = sqrt(pbar * (1 - pbar)))


### Q9: Repeat Q8 using an $np$ chart instead. Is your conclusion consistent with Q8?

The phase2 np chart also sees week2 as below the lower control limit. This is consistent with the p chart we used above and reinforces that the packaging improvement may have reduced the number of damaged comics. Both charts suggest an improvement, but continuous monitoring will help determine if the improvement is sustained over time. The point is technically out of control being lower than the control limit but in this case that is a favorable result with fewer comics being damaged.

In [ ]:
## Q9 Code ##

# phase2 np chart
np_phase2 <- qcc(phase2$NumberNonconforming,
                 sizes = phase2$TotalComics,
                 type = "np",
                 center = pbar,
                 std.dev = sqrt(pbar * (1 - pbar)))


### Q10: Read the New-Release Defect Counts data (Dataset B) into R. Construct a $c$ chart. Does the chart show evidence of statistical control? If not, identify the out-of-control week(s) by number and provide a contextual explanation as to why these OOC point(s) may have been observed, omit them, and replot the chart until statistical control is observed.

The original c chart is showing that the process is not in statistical control because week8 is above the upper limit with 52 defects. It's possible this was caused by a damaged shipment or improper loading during delivery. After we remove week8 and recalculate the control limits, the rest of the weeks fall within the limits, which indicate that the process is statistically in control.

In [ ]:
## Q10 Code ##

# data
newrelease <- read_excel("C:\\Users\\diirt\\Desktop\\STAT7110\\HW3RavisComics_NewReleaseDefectCounts.xlsx")

# original c chart
c1 <- qcc(newrelease$NumberOfDefects,
          type = "c")

# remove Week 8
newrelease2 <- newrelease %>%
  filter(Week != 8)

# update c chart
c2 <- qcc(newrelease2$NumberOfDefects,
          type = "c")


### Q11: Using your finalized $c$ chart, report $\bar{c}$ and the control limits. In 1-2 sentences, explain what this chart tells Ravi that the Q4-Q9 pass/fail analysis could not.

The finalized c chart has an average of 9.167 defects per five boxes with a lower control limit of .084 and an upper control limit of 18.250. Compared to the pass/fail analysis, the c chart measures the total number of individual defects which allows Ravi to better understand the overall damage since one comic can have multiple defects to it.

In [ ]:
## Q11 Code ##

# final center line and limits
cbar <- mean(newrelease2$NumberOfDefects)

LCL <- max(0, cbar - 3 * sqrt(cbar))
UCL <- cbar + 3 * sqrt(cbar)

cbar
LCL
UCL


### Q12: Read the Back-Issue Defect Counts data (Dataset C) into R. Construct a $u$ chart. Does the chart show evidence of statistical control? If not, identify the out-of-control week(s) by number and provide a contextual explanation as to why these OOC point(s) may have been observed, omit them, and replot the chart until statistical control is observed.

The original u chart shows that the process is not statistically in control because week 17 is exceeding the upper control limit with 34 defects across six boxes. This could be causedby improper storage, rough handling, or damage to a specific selection of back issue comics. After removing week17 and recalculating control limits, the remaining weeks fall within the limits which indicates that the process is in statistical control.

In [ ]:
## Q12 Code ##

# data
backissue <- read_excel("C:\\Users\\diirt\\Desktop\\STAT7110\\HW3RavisComics_BackIssueDefectCounts.xlsx")

# original u chart
u1 <- qcc(backissue$NumberOfDefects,
          sizes = backissue$BoxesInspected,
          type = "u")

# remove Week 17
backissue2 <- backissue %>%
  filter(Week != 17)

# update u chart
u2 <- qcc(backissue2$NumberOfDefects,
          sizes = backissue2$BoxesInspected,
          type = "u")


### Q13: Suppose you tried to build a $c$ chart on the Back-Issue Defect Counts data instead of a $u$ chart. Why would this not be an appropriate choice, given how the data were collected?

A c chart doesn't work well because the number of boxes inspected changes each week. A c chart assumes a constant inspection size so the weeks with more boxes would typically have more defects. The u chart is better fitting because it measures the average number of defects per box and adjusts the control limits based on the number of boxes that are inspected.

### Q14: Provide an overview of your results that Ravi can use the next time he calls the distributor's account representative. Citing specific evidence from your $p$/$np$ analysis **and** your $c$/$u$ analysis, which parts of the damage problem look like they're the distributor's responsibility, and which parts (if any) look more like an internal, in-shop handling issue?

The results suggest the bath the distributor and Ravi's shop could be contributing to the damage problem. The phase1 p and np charts identify week14 as out of control, suggesting there was an abnormal amount of damage in a shipment from the distributor. After the cardboard stiffeners were added, the nonconforming rate decreased from 5.16% to 2.71% and week2 of phase2 fell below the lower control limit. That tells us that the packaging improvement may have aided in reducing shipping damage. The new-release c chart identified week8 as out of control having 52 defects which suggests there was an unusally damaged shipment that may have resulted from packaging or transporation flaws. The back-issue u chart indentifies week17 as out of control with 34 defects within 6 boxes. Since these comics were already on the shelf, this suggests improper storage, handling, or customer engagement could be contributing to damage within the shop. Overall we think Ravi should continue working with the distributor to improve packaging while also reviewing how back-issue comics are cared for within the store.